<a href="https://colab.research.google.com/github/kkorotkovuky/bch612-colab/blob/main/AlphaFold_analysis_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyzing AlphaFold 3 predictions of a protein complex (BCH612, 2026)
AlphaFold 3 returns five models per job and a set of confidence files for each.
This notebook collects them, computes interface scores, compares models from different seeds, and, if a structure of the complex is known, measures how close each model is to it.

It reads two kinds of input:
- **AlphaFold Server** downloads (one `.zip` file per job), and
- **AlphaFold 3 run locally** (zip the output folder of the job first).

The worked example is the multidrug transporter NorM from *Vibrio cholerae* bound to the nanobody Nb17_4, compared with the cryo-EM structure 7PHP (Bloch *et al.*, 2021).

### Steps
1. A CPU runtime is enough (**Runtime → Change runtime type → CPU**).
2. Run the cells in order. Each cell has a ▶ button. Double-click a cell title to see its code.
3. Part 1 collects the scores, Part 2 compares seeds and samples, Part 3 compares the models with a known structure (optional).

In [ ]:
#@title 1. Install Biopython and load the analysis functions (about 20 s)
#@markdown Run once per session. All functions used below are defined in this cell.
import importlib.util, subprocess, sys
if importlib.util.find_spec("Bio") is None:
  subprocess.run([sys.executable, "-m", "pip", "-q", "install", "biopython"], check=True)

# Helper functions for the BCH612 AlphaFold analysis notebook.
# Reads AlphaFold Server zips and local AlphaFold 3 output folders, collects confidence
# scores, runs ipsae.py (DunbrackLab/IPSAE, pinned commit), and compares models with a
# reference structure (DockQ components, TM-score, interface residues).
import glob, json, os, re, subprocess, sys, urllib.request, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from Bio import Align
from Bio.PDB import MMCIFParser
from Bio.PDB.Polypeptide import is_aa, protein_letters_3to1

warnings.filterwarnings("ignore")
_PARSER = MMCIFParser(QUIET=True)
_ALIGNER = Align.PairwiseAligner(mode="global", open_gap_score=-11, extend_gap_score=-1,
                                 substitution_matrix=Align.substitution_matrices.load("BLOSUM62"))
BACKBONE = ("N", "CA", "C", "O")
IPSAE_URL = ("https://raw.githubusercontent.com/DunbrackLab/IPSAE/"
             "6174cf9e71cb1bd660cc805856a18c4871a6dec3/ipsae.py")


# ----------------------------------------------------------------------------------
# 1. Finding models in uploaded files
# ----------------------------------------------------------------------------------
def unpack_zips(zip_paths, dest):
    """Extract each zip into its own folder under dest; return the folders."""
    out = []
    for zp in zip_paths:
        d = Path(dest) / Path(zp).stem
        d.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp) as z:
            z.extractall(d)
        out.append(d)
    return out


def _seed_from_request(path):
    try:
        req = json.load(open(path))
        req = req[0] if isinstance(req, list) else req
        seeds = req.get("modelSeeds") or []
        return str(seeds[0]) if seeds else "unknown"
    except Exception:
        return "unknown"


def find_models(root):
    """Return one row per model found under root.

    AlphaFold Server: <job>_model_N.cif + <job>_full_data_N.json + <job>_summary_confidences_N.json;
    one job = one seed, N = 0-4 are the five diffusion samples (model_0 is ranked first).
    Local AlphaFold 3: <job>/seed-S_sample-K/<job>_seed-S_sample-K_{model.cif,confidences.json,
    summary_confidences.json} (older versions: model.cif, confidences.json, summary_confidences.json).
    """
    rows = []
    root = Path(root)
    # --- AlphaFold Server layout ---
    for s in sorted(root.rglob("*_summary_confidences_*.json")):
        m = re.match(r"(.+)_summary_confidences_(\d+)\.json$", s.name)
        if not m:
            continue
        job, n = m.group(1), int(m.group(2))
        cif, pae = s.with_name(f"{job}_model_{n}.cif"), s.with_name(f"{job}_full_data_{n}.json")
        if cif.exists() and pae.exists():
            rows.append(dict(job=job, source="AlphaFold Server",
                             seed=_seed_from_request(s.with_name(f"{job}_job_request.json")),
                             sample=n, cif=str(cif), pae_json=str(pae), summary_json=str(s)))
    # --- local AlphaFold 3 layout ---
    for d in sorted(root.rglob("seed-*_sample-*")):
        if not d.is_dir():
            continue
        m = re.match(r"seed-(\d+)_sample-(\d+)$", d.name)
        if not m:
            continue
        cif = next(iter(sorted(d.glob("*model.cif"))), None)
        conf = [p for p in sorted(d.glob("*confidences.json")) if "summary" not in p.name]
        summ = next(iter(sorted(d.glob("*summary_confidences.json"))), None)
        if cif is None or not conf or summ is None:
            continue
        job = d.parent.name
        dj = sorted(d.parent.glob("*_data.json"))
        if dj:
            job = dj[0].name[:-len("_data.json")]
        rows.append(dict(job=job, source="AlphaFold 3 (local)", seed=m.group(1),
                         sample=int(m.group(2)), cif=str(cif), pae_json=str(conf[0]),
                         summary_json=str(summ)))
    df = pd.DataFrame(rows)
    if len(df):
        df = df.drop_duplicates("cif").sort_values(["job", "seed", "sample", "cif"]).reset_index(drop=True)
        # the same job name in two uploads (e.g. a rerun): label each copy by its folder
        folder = df.cif.map(lambda p: Path(p).relative_to(root).parts[0])
        dup = df.duplicated(["job", "seed", "sample"], keep=False)
        df.loc[dup, "job"] = df.job[dup] + " (" + folder[dup] + ")"
        df.insert(0, "model_id", [f"{j} | seed {s} | sample {k}" for j, s, k in
                                  zip(df.job, df.seed, df["sample"])])
    return df


# ----------------------------------------------------------------------------------
# 2. Confidence scores
# ----------------------------------------------------------------------------------
def read_summary(path):
    s = json.load(open(path))
    return dict(ranking_score=s.get("ranking_score"), ptm=s.get("ptm"), iptm=s.get("iptm"),
                fraction_disordered=s.get("fraction_disordered"), has_clash=s.get("has_clash"),
                chain_pair_iptm=s.get("chain_pair_iptm"))


def get_ipsae_script(path="ipsae.py"):
    if not os.path.exists(path):
        urllib.request.urlretrieve(IPSAE_URL, path)
    return os.path.abspath(path)


def run_ipsae(pae_json, cif, script, pae_cutoff=10, dist_cutoff=10):
    """Run ipsae.py and return its per-direction table (one row per ordered chain pair)."""
    stem = cif[:-4]
    out = f"{stem}_{int(pae_cutoff):d}_{int(dist_cutoff):d}.txt"
    if not os.path.exists(out):
        subprocess.run([sys.executable, script, pae_json, cif, str(pae_cutoff), str(dist_cutoff)],
                       check=True, capture_output=True, text=True)
    lines = [l.split() for l in open(out) if l.strip()]
    df = pd.DataFrame(lines[1:], columns=lines[0])
    df = df[df.Type == "asym"].copy()
    for c in ["ipSAE", "pDockQ", "pDockQ2", "LIS", "ipTM_af", "n0res", "d0res"]:
        df[c] = pd.to_numeric(df[c])
    return df


def score_models(models, chain1="A", chain2="B", pae_cutoff=10, dist_cutoff=10,
                 script="ipsae.py"):
    """One row per model: AlphaFold's own scores plus ipSAE (both directions), pDockQ, pDockQ2, LIS."""
    script = get_ipsae_script(script)
    rows = []
    for _, m in models.iterrows():
        r = dict(model_id=m.model_id, job=m.job, seed=m.seed, sample=m["sample"])
        s = read_summary(m.summary_json)
        r.update({k: v for k, v in s.items() if k != "chain_pair_iptm"})
        try:
            ips = run_ipsae(m.pae_json, m.cif, script, pae_cutoff, dist_cutoff)
        except subprocess.CalledProcessError as e:
            print(f"ipsae.py failed for {m.model_id}: {e.stderr[-300:]}")
            rows.append(r)
            continue
        fwd = ips[(ips.Chn1 == chain1) & (ips.Chn2 == chain2)]
        rev = ips[(ips.Chn1 == chain2) & (ips.Chn2 == chain1)]
        if len(fwd) and len(rev):
            fwd, rev = fwd.iloc[0], rev.iloc[0]
            r["ipSAE_1to2"] = fwd.ipSAE   # aligned on chain1, scoring chain2
            r["ipSAE_2to1"] = rev.ipSAE   # aligned on chain2, scoring chain1
            r["pDockQ"] = fwd.pDockQ
            r["pDockQ2"] = max(fwd.pDockQ2, rev.pDockQ2)
            r["LIS"] = (fwd.LIS + rev.LIS) / 2
            r["n0res_1to2"], r["n0res_2to1"] = int(fwd.n0res), int(rev.n0res)
        rows.append(r)
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------
# 3. Structures: residues, contacts, PAE
# ----------------------------------------------------------------------------------
def load_structure(cif):
    return _PARSER.get_structure("s", cif)[0]


def protein_chains(model):
    """{chain_id: [residues with CA]} for protein chains only."""
    out = {}
    for ch in model:
        res = [r for r in ch if is_aa(r, standard=False) and "CA" in r]
        if len(res) >= 5:
            out[ch.id] = res
    return out


def seq_of(residues):
    return "".join(protein_letters_3to1.get(r.get_resname().capitalize(),
                   protein_letters_3to1.get(r.get_resname().upper(), "X")) for r in residues)


def _heavy(res):
    return [a for a in res if a.element != "H"]


def contact_pairs(res_a, res_b, cutoff=5.0):
    """Residue index pairs (i, j) with any heavy-atom distance below cutoff (angstrom)."""
    xa, ia = [], []
    for i, r in enumerate(res_a):
        for a in _heavy(r):
            xa.append(a.coord); ia.append(i)
    xb, ib = [], []
    for j, r in enumerate(res_b):
        for a in _heavy(r):
            xb.append(a.coord); ib.append(j)
    if not xa or not xb:
        return set()
    hits = cKDTree(np.array(xa)).query_ball_tree(cKDTree(np.array(xb)), cutoff)
    return {(ia[k], ib[j]) for k, js in enumerate(hits) for j in js}


def interface_residues(res_a, res_b, cutoff=5.0):
    p = contact_pairs(res_a, res_b, cutoff)
    return sorted({i for i, _ in p}), sorted({j for _, j in p})


def read_pae(pae_json):
    d = json.load(open(pae_json))
    return np.array(d["pae"], dtype=float), np.array(d["token_chain_ids"]), np.array(d["token_res_ids"])


def buried_area(model, chain1, chain2):
    """Buried solvent-accessible surface area (A^2) = SASA(1) + SASA(2) - SASA(1+2)."""
    from Bio.PDB.SASA import ShrakeRupley
    from Bio.PDB.Structure import Structure
    from Bio.PDB.Model import Model
    sr = ShrakeRupley()

    def sasa(chains):
        s = Structure("x"); m = Model(0); s.add(m)
        for c in chains:
            m.add(c.copy())
        sr.compute(m, level="M")
        return m.sasa

    a, b = model[chain1], model[chain2]
    return sasa([a]) + sasa([b]) - sasa([a, b])


# ----------------------------------------------------------------------------------
# 4. Reference comparison
# ----------------------------------------------------------------------------------
def fetch_pdb(pdb_id, dest="reference"):
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{pdb_id.upper()}.cif")
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.cif", path)
    return path


def residue_pairs(ref_res, mdl_res):
    """Sequence-align two residue lists; return [(ref_index, model_index)] for aligned positions."""
    aln = _ALIGNER.align(seq_of(ref_res), seq_of(mdl_res))[0]
    out = []
    for (rs, re_), (ms, me) in zip(aln.aligned[0], aln.aligned[1]):
        out.extend((rs + k, ms + k) for k in range(re_ - rs))
    return out


def match_chains(model, ref, min_identity=0.5):
    """Pair each model protein chain with the most similar unused reference chain."""
    mch, rch = protein_chains(model), protein_chains(ref)
    cand = []
    for mid, mres in mch.items():
        for rid, rres in rch.items():
            pairs = residue_pairs(rres, mres)
            ms, rs = seq_of(mres), seq_of(rres)
            ident = sum(rs[i] == ms[j] for i, j in pairs)
            cand.append(dict(model_chain=mid, ref_chain=rid, identical=ident,
                             identity=ident / max(len(pairs), 1),
                             coverage_of_ref=ident / len(rres), model_len=len(mres),
                             ref_len=len(rres)))
    cand = sorted(cand, key=lambda c: (-c["identity"] * c["coverage_of_ref"], -c["identical"]))
    used_m, used_r, out = set(), set(), []
    for c in cand:
        if c["model_chain"] in used_m or c["ref_chain"] in used_r or c["identity"] < min_identity:
            continue
        used_m.add(c["model_chain"]); used_r.add(c["ref_chain"]); out.append(c)
    return pd.DataFrame(out).sort_values("model_chain").reset_index(drop=True)


def kabsch(P, Q):
    """Rotation R and translation t that best map P onto Q (both N x 3)."""
    pc, qc = P.mean(0), Q.mean(0)
    H = (P - pc).T @ (Q - qc)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return R, qc - R @ pc


def _rmsd_after_fit(P, Q):
    R, t = kabsch(P, Q)
    return float(np.sqrt((((P @ R.T + t) - Q) ** 2).sum(1).mean()))


def tm_score(P, Q, L_ref):
    """TM-score of model CA coordinates P against reference CA coordinates Q (paired rows),
    normalized by the reference length L_ref. The superposition that maximizes TM-score is
    searched as in the TM-score program (Zhang & Skolnick 2004): start from fragments of
    decreasing length, then iteratively refit on residues closer than d0_search."""
    n = len(P)
    if n < 3:
        return 0.0
    d0 = 1.24 * (L_ref - 15) ** (1 / 3) - 1.8 if L_ref > 21 else 0.5
    d0_search = min(max(d0, 4.5), 8.0)
    best = 0.0
    L_init = n
    lengths = []
    while L_init >= 4:
        lengths.append(L_init)
        L_init //= 2
    if 4 not in lengths and n >= 4:
        lengths.append(4)
    for L in lengths:
        step = max(1, L // 2) if L < n else 1
        for start in range(0, n - L + 1, step):
            idx = np.arange(start, start + L)
            for _ in range(20):
                R, t = kabsch(P[idx], Q[idx])
                d = np.linalg.norm(P @ R.T + t - Q, axis=1)
                tm = float(np.sum(1 / (1 + (d / d0) ** 2)) / L_ref)
                best = max(best, tm)
                cut = d0_search
                new = np.where(d < cut)[0]
                while len(new) < 3 and cut < 20:
                    cut += 0.5
                    new = np.where(d < cut)[0]
                if len(new) < 3 or np.array_equal(new, idx):
                    break
                idx = new
    return best


def _bb(res_list, idxs):
    return [(k, a) for k in idxs for a in BACKBONE if a in res_list[k]]


def dockq_pair(m_rec, m_lig, r_rec, r_lig, rec_pairs, lig_pairs):
    """DockQ and its components for one interface (receptor = longer reference chain).

    fnat: fraction of reference residue-residue contacts (heavy atoms < 5 A) kept in the model.
    iRMSD: backbone RMSD over reference interface residues (heavy atoms < 10 A of the partner),
           after superposing those residues.
    LRMSD: ligand backbone RMSD after superposing the receptor backbone.
    DockQ = (fnat + 1/(1+(iRMSD/1.5)^2) + 1/(1+(LRMSD/8.5)^2)) / 3  (Basu & Wallner 2016)."""
    rmap = dict(rec_pairs)   # ref index -> model index
    lmap = dict(lig_pairs)
    nat = contact_pairs(r_rec, r_lig, 5.0)
    nat = {(i, j) for i, j in nat if i in rmap and j in lmap}
    mdl = contact_pairs(m_rec, m_lig, 5.0)
    kept = sum((rmap[i], lmap[j]) in mdl for i, j in nat)
    fnat = kept / len(nat) if nat else float("nan")

    def coords(res_list, pairs_map, ref_list, ref_idx):
        P, Q = [], []
        for i in ref_idx:
            mi = pairs_map[i]
            for a in BACKBONE:
                if a in ref_list[i] and a in res_list[mi]:
                    Q.append(ref_list[i][a].coord); P.append(res_list[mi][a].coord)
        return np.array(P), np.array(Q)

    # interface residues at 10 A for iRMSD
    i10_rec, i10_lig = interface_residues(r_rec, r_lig, 10.0)
    P1, Q1 = coords(m_rec, rmap, r_rec, [i for i in i10_rec if i in rmap])
    P2, Q2 = coords(m_lig, lmap, r_lig, [j for j in i10_lig if j in lmap])
    irmsd = _rmsd_after_fit(np.vstack([P1, P2]), np.vstack([Q1, Q2]))
    # LRMSD
    Pr, Qr = coords(m_rec, rmap, r_rec, sorted(rmap))
    Pl, Ql = coords(m_lig, lmap, r_lig, sorted(lmap))
    R, t = kabsch(Pr, Qr)
    lrmsd = float(np.sqrt((((Pl @ R.T + t) - Ql) ** 2).sum(1).mean()))
    dockq = (fnat + 1 / (1 + (irmsd / 1.5) ** 2) + 1 / (1 + (lrmsd / 8.5) ** 2)) / 3
    return dict(DockQ=dockq, fnat=fnat, iRMSD=irmsd, LRMSD=lrmsd, n_ref_contacts=len(nat))


def compare_to_reference(cif, ref, chain_map):
    """Per-chain TM-score and per-interface DockQ of one model.

    chain_map: {model_chain: ref_chain}. Returns (chain_rows, interface_rows)."""
    mdl = load_structure(cif)
    mch, rch = protein_chains(mdl), protein_chains(ref)
    pairs = {mc: residue_pairs(rch[rc], mch[mc]) for mc, rc in chain_map.items()}
    chain_rows = []
    for mc, rc in chain_map.items():
        P = np.array([mch[mc][j]["CA"].coord for _, j in pairs[mc]])
        Q = np.array([rch[rc][i]["CA"].coord for i, _ in pairs[mc]])
        chain_rows.append(dict(model_chain=mc, ref_chain=rc, aligned=len(P),
                               ref_residues=len(rch[rc]),
                               TM_score=tm_score(P, Q, len(rch[rc])),
                               CA_RMSD=_rmsd_after_fit(P, Q)))
    iface_rows = []
    mcs = list(chain_map)
    for a in range(len(mcs)):
        for b in range(a + 1, len(mcs)):
            ma, mb = mcs[a], mcs[b]
            ra, rb = chain_map[ma], chain_map[mb]
            if not contact_pairs(rch[ra], rch[rb], 5.0):
                continue   # these two chains do not touch in the reference
            # receptor = longer chain in the reference
            if len(rch[ra]) < len(rch[rb]):
                ma, mb, ra, rb = mb, ma, rb, ra
            q = dockq_pair(mch[ma], mch[mb], rch[ra], rch[rb], pairs[ma], pairs[mb])
            iface_rows.append(dict(receptor=f"{ma} (ref {ra})", ligand=f"{mb} (ref {rb})",
                                   rec_chain=ma, lig_chain=mb, **q))
    return pd.DataFrame(chain_rows), pd.DataFrame(iface_rows)


def interface_comparison(cif, ref, chain_map, chain1, chain2, cutoff=5.0):
    """Interface residues of chain1 and chain2 in the model and in the reference,
    in reference numbering: shared, model only, reference only."""
    mdl = load_structure(cif)
    mch, rch = protein_chains(mdl), protein_chains(ref)
    out = []
    m_if = interface_residues(mch[chain1], mch[chain2], cutoff)
    r_if = interface_residues(rch[chain_map[chain1]], rch[chain_map[chain2]], cutoff)
    for side, mc, mi, ri in ((0, chain1, m_if[0], r_if[0]), (1, chain2, m_if[1], r_if[1])):
        rc = chain_map[mc]
        m2r = {j: i for i, j in residue_pairs(rch[rc], mch[mc])}
        mi_ref = {m2r[j] for j in mi if j in m2r}
        ri = set(ri)
        lab = lambda i: (f"{rch[rc][i].get_resname().capitalize()}{rch[rc][i].id[1]}"
                         f"{rch[rc][i].id[2].strip()}")
        shared, monly, ronly = sorted(mi_ref & ri), sorted(mi_ref - ri), sorted(ri - mi_ref)
        out.append(dict(chain=f"{mc} (ref {rc})", reference_interface=len(ri),
                        model_interface=len(mi_ref), shared=len(shared),
                        recall=len(shared) / max(len(ri), 1),
                        precision=len(shared) / max(len(mi_ref), 1),
                        shared_residues=" ".join(lab(i) for i in shared),
                        model_only=" ".join(lab(i) for i in monly),
                        reference_only=" ".join(lab(i) for i in ronly)))
    return pd.DataFrame(out)


from IPython.display import display
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
IPSAE = get_ipsae_script("ipsae.py")
print("Ready. ipsae.py from DunbrackLab/IPSAE, commit 6174cf9 (version 4).")

## Part 1. Collect the models and their scores

In [ ]:
#@title 2. Load AlphaFold results
#@markdown Check `use_demo_data` to analyze the worked example: NorM with Nb17_4, one AlphaFold Server job with templates and five jobs without templates (30 models).
#@markdown Uncheck it to upload your own `.zip` files: AlphaFold Server downloads or a zipped local AlphaFold 3 output folder. You can select several files at once.
use_demo_data = True #@param {type:"boolean"}
import os, shutil, urllib.request
DEMO_BASE = os.environ.get("BCH612_DEMO_BASE",
  "https://github.com/kkorotkovuky/bch612-colab/releases/download/norm-demo-v1/")
DEMO_ZIPS = ["fold_norm_nb17_4.zip"] + ["fold_norm_nb17_4_no_templates.zip"] + \
            [f"fold_norm_nb17_4_no_templates_{i}.zip" for i in range(2, 6)]

os.makedirs("uploads", exist_ok=True)
if use_demo_data:
  zips = []
  for z in DEMO_ZIPS:
    p = os.path.join("uploads", z)
    if not os.path.exists(p):
      print("Downloading", z)
      urllib.request.urlretrieve(DEMO_BASE + z, p)
    zips.append(p)
else:
  from google.colab import files
  uploaded = files.upload()
  zips = []
  for name, data in uploaded.items():
    if not name.lower().endswith(".zip"):
      print(f"Skipping {name}: please upload .zip files")
      continue
    p = os.path.join("uploads", name)
    open(p, "wb").write(data)
    zips.append(p)

JOBS_DIR = "af_jobs"
shutil.rmtree(JOBS_DIR, ignore_errors=True)
unpack_zips(zips, JOBS_DIR)
MODELS = find_models(JOBS_DIR)
if len(MODELS) == 0:
  raise RuntimeError("No AlphaFold models found in the uploaded files.")
print(f"Found {len(MODELS)} models in {MODELS.job.nunique()} jobs:")
display(MODELS.groupby(["job", "source", "seed"], sort=False).size().rename("models").reset_index())

### Where the scores live
For every model, AlphaFold writes three files. In an AlphaFold Server download, model `N` (0–4) of a job called `fold_myjob` gives:

| File | Contents |
|---|---|
| `fold_myjob_model_N.cif` | The coordinates. The B-factor column holds the per-atom **pLDDT** (0–100). |
| `fold_myjob_summary_confidences_N.json` | A few numbers for the whole model: `ptm`, `iptm`, `ranking_score`, `chain_pair_iptm`, `fraction_disordered`, `has_clash`. |
| `fold_myjob_full_data_N.json` | The full **PAE** matrix (predicted aligned error) and the per-atom pLDDT values. |

A local AlphaFold 3 run puts the same three files for each model in a folder called `seed-S_sample-K`, with names ending in `_model.cif`, `_summary_confidences.json`, and `_confidences.json`.

Open the **Files** panel on the left (folder icon) and look inside `af_jobs` to see them.

In [ ]:
#@title 3. Exercise: find ipTM in a summary file yourself
#@markdown Run this cell to print the summary file of the first model. Find the value of `iptm`, type it into `my_iptm`, and run the cell again.
my_iptm = 0.0 #@param {type:"number"}
import json
row = MODELS.iloc[0]
s = json.load(open(row.summary_json))
print(os.path.relpath(row.summary_json, JOBS_DIR), "\n")
for k, v in s.items():
  if k == "chain_ids":
    print(f"  {k}: [... {len(v)} entries, one chain letter per token ...]")
  else:
    print(f"  {k}: {v}")
print()
if my_iptm == 0:
  print("Type the ipTM value into my_iptm and run the cell again.")
elif abs(my_iptm - s["iptm"]) < 0.005:
  print("Correct.")
  print("chain_pair_iptm is a table with one row and one column per chain.")
  print("Off the diagonal: ipTM of each pair of chains. On the diagonal: the pTM of each chain alone.")
else:
  print("Not quite. Look for the line that starts with 'iptm' (not 'chain_iptm').")

### Interface scores
AlphaFold reports **pTM** and **ipTM**: predicted TM-scores for the whole complex and for its interfaces, both calculated from the PAE matrix.
The **ranking score** that AlphaFold uses to order the five models is 0.8 × ipTM + 0.2 × pTM + 0.5 × fraction disordered − 100 × has clash.
The Server rounds these numbers to two decimals, so ties are common.

Several other interface scores have been proposed. Step 4 computes them with `ipsae.py` (Dunbrack, 2025):

| Score | What it uses |
|---|---|
| **ipSAE** | Like ipTM, but counts only residue pairs whose PAE is below a cutoff (10 Å by default), and normalizes by the number of residues that pass. Large unstructured regions therefore do not pull it down. |
| **pDockQ** | pLDDT of residues at the interface and the number of interface contacts (Bryant *et al.*, 2022). |
| **pDockQ2** | Interface pLDDT combined with PAE (Zhu *et al.*, 2023). |
| **LIS** | Average of (12 − PAE)/12 over the inter-chain pairs with PAE ≤ 12 Å (Kim *et al.*, 2024). |
| **Buried area** | Solvent-accessible surface area hidden in the interface (Å²): SASA(chain 1) + SASA(chain 2) − SASA(complex). |

**PAE has a direction.** PAE(*i*, *j*) is the expected error in the position of residue *j* when the model is superposed on residue *i*.
So the error of chain 2 measured from chain 1 (*aligned on chain 1, scoring chain 2*) need not equal the reverse.
ipSAE is reported for both directions. Like TM-score, it also depends on how many residues are counted, so the two directions can differ even when the PAE values are similar.
When you compare models, keep the direction fixed. pDockQ2 is reported once per pair as the larger of its two directions, and LIS as their mean, as in `ipsae.py`.

In [ ]:
#@title 4. Score every model
#@markdown Choose the two chains whose interface you want to score (chain letters as in the model files).
chain_1 = "A" #@param {type:"string"}
chain_2 = "B" #@param {type:"string"}
PAE_cutoff = 10 #@param {type:"number"}
distance_cutoff = 10 #@param {type:"number"}
compute_buried_area = True #@param {type:"boolean"}

chains_found = sorted(protein_chains(load_structure(MODELS.cif[0])))
for c in (chain_1, chain_2):
  if c not in chains_found:
    raise ValueError(f"Chain {c} not found. Protein chains in the first model: {chains_found}")

print(f"Running ipsae.py on {len(MODELS)} models ...")
SCORES = score_models(MODELS, chain_1, chain_2, PAE_cutoff, distance_cutoff, IPSAE)
if compute_buried_area:
  print("Computing buried surface area ...")
  SCORES["buried_area"] = [buried_area(load_structure(c), chain_1, chain_2) for c in MODELS.cif]

LABELS = {"ranking_score": "ranking score", "ptm": "pTM", "iptm": "ipTM",
          "ipSAE_1to2": f"ipSAE (align {chain_1}, score {chain_2})",
          "ipSAE_2to1": f"ipSAE (align {chain_2}, score {chain_1})",
          "pDockQ": "pDockQ", "pDockQ2": "pDockQ2", "LIS": "LIS", "buried_area": "buried area (Å²)"}
show = SCORES[["job", "seed", "sample"] + [k for k in LABELS if k in SCORES]].rename(columns=LABELS)
display(show.round(3))

### Look inside one model
The next cell plots the PAE matrix of one model and lists the residues at the interface.
The rows of the matrix are the residues the model is aligned on; the columns are the residues whose error is shown.
The two off-diagonal blocks are the two directions of the same interface.

In [ ]:
#@title 5. PAE map and interface residues of one model
#@markdown `model_row` is the row number in the table of Step 4 (0 = first row).
model_row = 0 #@param {type:"integer"}
m = MODELS.iloc[model_row]
pae, tok_chain, tok_res = read_pae(m.pae_json)

in_1, in_2 = tok_chain == chain_1, tok_chain == chain_2
block_12 = pae[np.ix_(in_1, in_2)]   # aligned on chain 1, error of chain 2 residues
block_21 = pae[np.ix_(in_2, in_1)]   # aligned on chain 2, error of chain 1 residues

fig, ax = plt.subplots(figsize=(6, 5.2))
im = ax.imshow(pae, cmap="Greens_r", vmin=0, vmax=31.75)
edges = [k for k in range(1, len(tok_chain)) if tok_chain[k] != tok_chain[k - 1]]
for e in edges:
  ax.axhline(e - 0.5, color="k", lw=0.8); ax.axvline(e - 0.5, color="k", lw=0.8)
starts = [0] + edges
centers = [(s0 + e0) / 2 for s0, e0 in zip(starts, edges + [len(tok_chain)])]
names = [f"chain {tok_chain[k]}" for k in starts]
top = ax.secondary_xaxis("top"); top.set_xticks(centers, labels=names); top.tick_params(length=0)
right = ax.secondary_yaxis("right"); right.set_yticks(centers, labels=names); right.tick_params(length=0)
ax.set_xlabel("Scored residue (j)"); ax.set_ylabel("Aligned residue (i)")
ax.set_title(m.model_id, fontsize=9, pad=22)
fig.colorbar(im, ax=ax, label="PAE (Å)", shrink=0.8, pad=0.12)
plt.show()

for name, blk in ((f"aligned on {chain_1}, scoring {chain_2}", block_12),
                  (f"aligned on {chain_2}, scoring {chain_1}", block_21)):
  print(f"{name}: mean PAE {blk.mean():.1f} Å, lowest {blk.min():.1f} Å, "
        f"{(blk < PAE_cutoff).mean() * 100:.1f}% of pairs below {PAE_cutoff} Å")

# residues in contact across the interface (any heavy atoms closer than 5 Å)
st = load_structure(m.cif)
ch = protein_chains(st)
idx = {(c, r): k for k, (c, r) in enumerate(zip(tok_chain, tok_res))}
if1, if2 = interface_residues(ch[chain_1], ch[chain_2], 5.0)
rows = []
for c, other, lst, res in ((chain_1, chain_2, if1, ch[chain_1]), (chain_2, chain_1, if2, ch[chain_2])):
  partner = np.array([k for (cc, _), k in idx.items() if cc == other])
  for i in lst:
    r = res[i]; k = idx[(c, r.id[1])]
    rows.append(dict(chain=c, residue=f"{r.get_resname().capitalize()}{r.id[1]}",
                     pLDDT=round(float(r["CA"].get_bfactor()), 1),
                     lowest_PAE_to_partner=round(float(min(pae[k, partner].min(), pae[partner, k].min())), 1)))
IFACE = pd.DataFrame(rows)
print(f"\nInterface residues (heavy atoms within 5 Å): {len(if1)} in chain {chain_1}, {len(if2)} in chain {chain_2}")
display(IFACE)

## Part 2. Seeds and diffusion samples
AlphaFold 3 has two sources of variation.
For each **seed**, it runs the main network (the trunk) once and then the diffusion module draws **five samples** from it.
A different seed changes the random choices made throughout the network.

- On the **AlphaFold Server**, each job uses one seed and returns its five samples as models 0–4. To get more seeds, submit the same input again; unless you fix the seed yourself, the Server picks a new one at random.
- In a **local AlphaFold 3** run, the seeds are listed in `modelSeeds` in the input file, and each `seed-S_sample-K` folder is one sample.

In [ ]:
#@title 6. Compare seeds and samples
score_to_compare = "ipTM" #@param ["ranking score", "ipTM", "pTM", "ipSAE (aligned on chain 1)", "ipSAE (aligned on chain 2)", "pDockQ", "pDockQ2", "LIS"]
col = {"ranking score": "ranking_score", "ipTM": "iptm", "pTM": "ptm",
       "ipSAE (aligned on chain 1)": "ipSAE_1to2", "ipSAE (aligned on chain 2)": "ipSAE_2to1",
       "pDockQ": "pDockQ", "pDockQ2": "pDockQ2", "LIS": "LIS"}[score_to_compare]
label = LABELS.get(col, score_to_compare)

groups = list(SCORES.groupby(["job", "seed"], sort=False))
fig, ax = plt.subplots(figsize=(max(5, 1.1 * len(groups)), 4))
rng = np.random.default_rng(0)
for x, ((job, seed), g) in enumerate(groups):
  ax.scatter(x + rng.uniform(-0.12, 0.12, len(g)), g[col], s=28, color="tab:blue", alpha=0.8)
  ax.plot([x - 0.25, x + 0.25], [g[col].mean()] * 2, color="k", lw=1.5)
ax.set_xticks(range(len(groups)))
ax.set_xticklabels([f"{job}\nseed {seed}" for (job, seed), _ in groups], rotation=45, ha="right", fontsize=8)
ax.set_ylabel(label); ax.set_title("Each dot is one sample; the bar is the seed mean", fontsize=10)
plt.tight_layout(); plt.show()

per_seed = SCORES.groupby(["job", "seed"], sort=False)[col].agg(samples="size", mean="mean", lowest="min", highest="max")
per_seed["range"] = per_seed.highest - per_seed.lowest
display(per_seed.round(3))
print(f"Average range within a seed: {per_seed['range'].mean():.3f}")
if len(per_seed) > 1:
  print(f"Range of the seed means:     {per_seed['mean'].max() - per_seed['mean'].min():.3f}")
else:
  print("Only one seed: submit the same input again with another seed to compare seeds.")
top = SCORES.loc[SCORES.ranking_score.idxmax()]
print(f"\nModel with the highest ranking score: {top.model_id} ({top.ranking_score:.2f})")

## Part 3. Compare with a known structure (optional)
If a structure of the complex has been determined, enter its PDB ID.
The model chains are matched with the reference chains by sequence, so the chain letters do not have to agree.

For each model, the next cells report:
- **TM-score** of each chain (0–1), normalized by the length of the reference chain. Values above 0.5 usually mean the same fold.
- **DockQ** (0–1) for each interface present in the reference, with its three components:
  **fnat**, the fraction of reference residue contacts (heavy atoms within 5 Å) that the model reproduces;
  **iRMSD**, the backbone RMSD of the interface residues (within 10 Å of the partner) after superposing them;
  **LRMSD**, the backbone RMSD of the smaller chain after superposing the larger one.
  DockQ = (fnat + 1/(1 + (iRMSD/1.5)²) + 1/(1 + (LRMSD/8.5)²)) / 3. The published quality classes are: below 0.23 incorrect, 0.23–0.49 acceptable, 0.49–0.80 medium, 0.80 and above high (Basu & Wallner, 2016).

In [ ]:
#@title 7. Download the reference structure and match chains
pdb_id = "7PHP" #@param {type:"string"}
#@markdown Leave `chain_pairs` empty to match chains by sequence, or set them yourself as `A:A, B:N` (model chain : reference chain).
chain_pairs = "" #@param {type:"string"}
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
ref_path = fetch_pdb(pdb_id)
REF = load_structure(ref_path)

# what each reference chain is, from the mmCIF entity records
d = MMCIF2Dict(ref_path)
desc = {}
for ids, name in zip(d.get("_entity_poly.pdbx_strand_id", []),
                     [d["_entity.pdbx_description"][d["_entity.id"].index(e)] for e in d.get("_entity_poly.entity_id", [])]):
  for c in ids.split(","):
    desc[c.strip()] = name

MATCH = match_chains(load_structure(MODELS.cif[0]), REF)
MATCH["reference_description"] = MATCH.ref_chain.map(desc)
if chain_pairs.strip():
  CHAIN_MAP = dict(p.strip().split(":") for p in chain_pairs.split(","))
else:
  CHAIN_MAP = dict(zip(MATCH.model_chain, MATCH.ref_chain))
print(f"{pdb_id}: chain pairs used {CHAIN_MAP}")
display(MATCH[["model_chain", "ref_chain", "reference_description", "identity", "coverage_of_ref", "model_len", "ref_len"]].round(3))
others = {c: desc.get(c, "") for c in protein_chains(REF) if c not in CHAIN_MAP.values()}
if others:
  print("Reference chains not used:", "; ".join(f"{c} ({n})" for c, n in others.items()))

In [ ]:
#@title 8. Score every model against the reference
rows, all_ifaces = [], []
for _, m in MODELS.iterrows():
  ch_df, if_df = compare_to_reference(m.cif, REF, CHAIN_MAP)
  r = {"model_id": m.model_id}
  for _, c in ch_df.iterrows():
    r[f"TM-score {c.model_chain}"] = c.TM_score
  for _, i in if_df.iterrows():
    i = i.to_dict(); i["model_id"] = m.model_id; all_ifaces.append(i)
  main = if_df[[{a, b} == {chain_1, chain_2} for a, b in zip(if_df.rec_chain, if_df.lig_chain)]]
  if len(main):
    r.update(main.iloc[0][["DockQ", "fnat", "iRMSD", "LRMSD"]].to_dict())
  rows.append(r)
ACC = pd.DataFrame(rows)
IFACES_REF = pd.DataFrame(all_ifaces)
RESULTS = SCORES.merge(ACC, on="model_id")
if "DockQ" in RESULTS:
  RESULTS["DockQ class"] = pd.cut(RESULTS.DockQ, [-1, 0.23, 0.49, 0.80, 1.01],
                                  labels=["incorrect", "acceptable", "medium", "high"], right=False)
cols = ["job", "seed", "sample"] + [c for c in ACC.columns if c != "model_id"] + (["DockQ class"] if "DockQ" in RESULTS else [])
display(RESULTS[cols].round(3))
if IFACES_REF.receptor.nunique() > 1 or IFACES_REF.ligand.nunique() > 1:
  print("All interfaces present in the reference:"); display(IFACES_REF.drop(columns=["rec_chain", "lig_chain"]).round(3))

In [ ]:
#@title 9. Does confidence predict accuracy?
x_score = "ipTM" #@param ["ranking score", "ipTM", "pTM", "ipSAE (aligned on chain 1)", "ipSAE (aligned on chain 2)", "pDockQ", "pDockQ2", "LIS", "buried area"]
xcol = {"ranking score": "ranking_score", "ipTM": "iptm", "pTM": "ptm",
        "ipSAE (aligned on chain 1)": "ipSAE_1to2", "ipSAE (aligned on chain 2)": "ipSAE_2to1",
        "pDockQ": "pDockQ", "pDockQ2": "pDockQ2", "LIS": "LIS", "buried area": "buried_area"}[x_score]
from scipy.stats import spearmanr
fig, ax = plt.subplots(figsize=(6, 4.2))
for job, g in RESULTS.groupby("job", sort=False):
  ax.scatter(g[xcol], g.DockQ, s=30, alpha=0.8, label=job)
for y, name in ((0.23, "acceptable"), (0.49, "medium"), (0.80, "high")):
  ax.axhline(y, color="grey", ls="--", lw=0.8)
  ax.text(ax.get_xlim()[0], y, f" {name}", va="bottom", ha="left", fontsize=8, color="grey")
ax.set_xlabel(LABELS.get(xcol, x_score)); ax.set_ylabel("DockQ"); ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(); plt.show()
rho, p = spearmanr(RESULTS[xcol], RESULTS.DockQ)
print(f"Spearman correlation between {x_score} and DockQ over {len(RESULTS)} models: {rho:.2f}")

In [ ]:
#@title 10. Interface residues: model versus reference
#@markdown Residues are named with the reference numbering. Recall = shared / reference interface; precision = shared / model interface.
model_row = 0 #@param {type:"integer"}
m = MODELS.iloc[model_row]
CMP = interface_comparison(m.cif, REF, CHAIN_MAP, chain_1, chain_2)
print(m.model_id, "\n")
for _, r in CMP.iterrows():
  print(f"Chain {r.chain}: {r.reference_interface} reference interface residues, {r.model_interface} in the model, "
        f"{r.shared} shared (recall {r.recall:.2f}, precision {r.precision:.2f})")
  print(f"  shared:         {r.shared_residues}")
  print(f"  model only:     {r.model_only or '-'}")
  print(f"  reference only: {r.reference_only or '-'}\n")

In [ ]:
#@title 11. Download the results table
out = "af_analysis_results.csv"
(RESULTS if "RESULTS" in globals() else SCORES).drop(columns=["model_id"]).to_csv(out, index=False)
print("Saved", out)
try:
  from google.colab import files
  files.download(out)
except ImportError:
  pass

## Questions
1. In Step 4, the two ipSAE directions differ. Which direction is higher in the worked example, and how many residues pass the PAE cutoff in each direction? (Hint: `SCORES[["n0res_1to2", "n0res_2to1"]]`.)
2. In Step 6, do the samples of one seed differ more or less than the seeds of one input? What does this suggest about how many seeds to run?
3. In Step 9, does the model with the highest ranking score also have the highest DockQ? Is there a value of any score above which every model is correct?
4. Compare the job with templates and the jobs without templates. Which scores separate them?
5. In Step 10, which reference interface residues does a poor model miss, and where are they on the protein?

## References
- Abramson J *et al.* (2024) Accurate structure prediction of biomolecular interactions with AlphaFold 3. *Nature* 630:493–500. https://doi.org/10.1038/s41586-024-07487-w
- Dunbrack RL (2025) Rēs ipSAE loquuntur: What's wrong with AlphaFold's ipTM score and how to fix it. *bioRxiv*. https://doi.org/10.1101/2025.02.10.637595 — code: https://github.com/DunbrackLab/IPSAE (MIT license)
- Bryant P, Pozzati G, Elofsson A (2022) Improved prediction of protein-protein interactions using AlphaFold2. *Nat Commun* 13:1265. https://doi.org/10.1038/s41467-022-28865-w
- Zhu W, Shenoy A, Kundrotas P, Elofsson A (2023) Evaluation of AlphaFold-Multimer prediction on multi-chain protein complexes. *Bioinformatics* 39:btad424. https://doi.org/10.1093/bioinformatics/btad424
- Kim A-R *et al.* (2024) Enhanced protein-protein interaction discovery via AlphaFold-Multimer. *bioRxiv*. https://doi.org/10.1101/2024.02.19.580970
- Basu S, Wallner B (2016) DockQ: a quality measure for protein-protein docking models. *PLoS One* 11:e0161879. https://doi.org/10.1371/journal.pone.0161879
- Mirabello C, Wallner B (2024) DockQ v2: improved automatic quality measure for protein multimers, nucleic acids, and small molecules. *Bioinformatics* 40:btae586. https://doi.org/10.1093/bioinformatics/btae586
- Zhang Y, Skolnick J (2004) Scoring function for automated assessment of protein structure template quality. *Proteins* 57:702–710. https://doi.org/10.1002/prot.20264
- Bloch JS *et al.* (2021) Development of a universal nanobody-binding Fab module for fiducial-assisted cryo-EM studies of membrane proteins. *Proc Natl Acad Sci USA* 118:e2115435118. https://doi.org/10.1073/pnas.2115435118

DockQ and TM-score are computed here with NumPy and Biopython from the published definitions; on the worked example they agree with DockQ 2.1.3 (within 0.001) and US-align (within 0.001).